In [1]:
def sample_precision_matrix(Y):
    """
    Computes the precision matrix using the sample covariance matrix.
    
    Parameters:
    -----------
    Y : np.ndarray, shape (n, p)
        Returns matrix (time x assets)
    
    Returns:
    --------
    Theta_hat : np.ndarray, shape (p, p)
        Estimated precision matrix (inverse of sample covariance)
    """
    # Calculate sample covariance matrix (ddof=1 for unbiased estimator)
    # rowvar=False means columns are assets and rows are time periods
    Sigma_hat = np.cov(Y, rowvar=False)
    
    # Calculate precision matrix as the inverse of the covariance matrix
    # (Using inv assumes the matrix is non-singular)
    Theta_hat = np.linalg.inv(Sigma_hat)
    
    return Theta_hat

def gmv_weights(Theta_hat):
    """
    Compute Global Minimum Variance (GMV) portfolio weights (Section 6.1).
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # w* = (Θ 1_p) / (1_p' Θ 1_p)
    numerator = Theta_hat @ ones_p
    denominator = ones_p @ Theta_hat @ ones_p
    
    if np.abs(denominator) < 1e-10:
        # Fallback to equal weights if precision matrix is near-singular
        return ones_p / p
    
    w_star = numerator / denominator
    
    return w_star

def mv_weights(Theta_hat, mu, target_variance=0.15**2/12):
    """
    Compute Mean-Variance portfolio weights to maximize return given a target variance.
    
    Solves the constrained optimization:
    max w' mu  subject to  w' Sigma w <= target_variance  and  w' 1 = 1
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected returns
    target_variance : float
        Target maximum portfolio variance
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute key quantities
    A = ones_p @ Theta_hat @ ones_p  # 1' Theta 1
    B = ones_p @ Theta_hat @ mu      # 1' Theta mu  
    C = mu @ Theta_hat @ mu          # mu' Theta mu
    D = A * C - B * B                # Determinant
    
    # Check for singularity
    if np.abs(D) < 1e-10 or np.abs(A) < 1e-10:
        print('SINGULARITY: Falling back to equal weights')
        return ones_p / p
        
    # Minimum possible variance (Global Minimum Variance)
    min_variance = 1.0 / A
    
    # Check feasibility
    if target_variance < min_variance:
        print(f'INFEASIBLE: Target variance {target_variance:.6f} is below GMV variance {min_variance:.6f}. Returning GMV portfolio.')
        return (Theta_hat @ ones_p) / A

    # Map target variance to maximum expected return on the efficient frontier
    # The efficient frontier equation is: variance = (A*mu^2 - 2*B*mu + C) / D
    # Solving for the upper half (max return) yields:
    target_return = (B + np.sqrt(D * (A * target_variance - 1))) / A
    
    # Compute Lagrange multipliers using the derived target_return
    lambda1 = (C - B * target_return) / D
    lambda2 = (A * target_return - B) / D
    
    # Compute weights: w = lambda1 * Theta^{-1} 1 + lambda2 * Theta^{-1} mu
    w_star = lambda1 * (Theta_hat @ ones_p) + lambda2 * (Theta_hat @ mu)
    
    return w_star

def msr_weights(Theta_hat, mu):
    """
    Compute Maximum Sharpe Ratio portfolio weights.
    
    The maximum Sharpe ratio portfolio solves:
    max (w' mu) / sqrt(w' Sigma w)
    
    Solution (when mu represents excess returns):
    w ∝ Sigma^{-1} mu = Theta mu
    
    Then normalize so that sum(w) = 1.
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected excess returns
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights (sum to 1)
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute unnormalized weights: w ∝ Theta mu
    w_unnorm = Theta_hat @ mu
    
    # Normalize to sum to 1
    weight_sum = np.sum(w_unnorm)
    
    if np.abs(weight_sum) < 1e-10:
        print('WARNING: Weight sum near zero, returning equal weights')
        return ones_p / p
    
    w_star = w_unnorm / weight_sum
    
    return w_star

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

def train_random_forest(df, train_start, train_end, features=['mom12m', 'mve', 'bm'], clip_tails=True):
    """
    Train Random Forest Regressor on historical data to predict return magnitude.
    """
    train_df = df[(df['datadate'] >= train_start) & (df['datadate'] <= train_end)].copy()
    
    # Drop rows missing features or the continuous target
    train_df = train_df.dropna(subset=features + ['ret_fwd_1'])
    
    # Winsorize (clip) extreme outliers to protect the Regressor's MSE loss function
    if clip_tails:
        lower_cap = train_df['ret_fwd_1'].quantile(0.01)
        upper_cap = train_df['ret_fwd_1'].quantile(0.99)
        train_df['target'] = train_df['ret_fwd_1'].clip(lower_cap, upper_cap)
    else:
        train_df['target'] = train_df['ret_fwd_1']
    
    X_train = train_df[features]
    y_train = train_df['target']
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Use Regressor instead of Classifier
    rf_model = RandomForestRegressor(
        n_estimators=200,      # More trees to smooth variance
        max_depth=8,           # Increased from 5 to allow deeper interactions
        min_samples_leaf=15,   # Decreased from 50 to allow smaller, more specific leaf nodes
        max_features='sqrt',   # Forces trees to use different features, increasing variance
        random_state=42, 
        n_jobs=-1
    )
    rf_model.fit(X_train_scaled, y_train)
    
    print(f"  Training samples: {len(train_df)}")
    print(f"  Training R-squared: {rf_model.score(X_train_scaled, y_train):.4f}")
    
    return rf_model, scaler


def select_stocks_rf_quantiles(df, predict_date, rf_model, scaler, 
                                         features=['mom12m', 'mve', 'bm'],
                                         top_q=0.90, bottom_q=0.10):
    """
    Select top and bottom percentiles of stocks based on predicted expected return.
    """
    predict_df = df[df['datadate'] == predict_date].copy()
    predict_df = predict_df.dropna(subset=features)
    
    if len(predict_df) == 0:
        print(f"  ⚠ No stocks with complete data on {predict_date}")
        return []
    
    X_predict = predict_df[features]
    X_predict_scaled = scaler.transform(X_predict)
    
    # Regressor uses .predict() to output a continuous return estimate, not a probability
    predict_df['expected_return'] = rf_model.predict(X_predict_scaled)
    
    # Calculate quantiles on the predicted magnitude
    upper_bound = predict_df['expected_return'].quantile(top_q)
    lower_bound = predict_df['expected_return'].quantile(bottom_q)
    
    # Screen out the dead zone
    selected_df = predict_df[
        (predict_df['expected_return'] >= upper_bound) | 
        (predict_df['expected_return'] <= lower_bound)
    ]
    
    selected_permnos = selected_df['permno'].tolist()
    
    print(f"  Stocks evaluated: {len(predict_df)}")
    print(f"  Upper expected return bound (Top {100-top_q*100:.0f}%): {upper_bound:.4f}")
    print(f"  Lower expected return bound (Bottom {bottom_q*100:.0f}%): {lower_bound:.4f}")
    print(f"  Stocks selected: {len(selected_permnos)}")
    
    return selected_permnos

In [3]:
def integrated_backtest(df,
                       test_start_date='2020-01-31',
                       test_end_date='2024-04-30',
                       rf_train_years=15,
                       features=['mom12m', 'mve', 'bm'],
                       top_q=0.90,
                       bottom_q=0.10,
                       lookback_window=180,
                       transaction_cost=0.005,
                       portfolio_type='all',  
                       mv_target_variance=0.15**2/12,
                       verbose=True):
    """
    Integrated backtest combining logistic regression stock selection 
    with nodewise regression portfolio optimization.
    
    Annual workflow:
    1. Train logistic regression on past 15 years (every January)
    2. Select stocks based on buy probability
    3. Run nodewise regression on selected stocks monthly until next retrain
    
    Parameters:
    -----------
    df : pd.DataFrame
        Full dataset with columns: datadate, permno, ret_fwd_1, mom12m, mve, bm
    test_start_date : str
        Start date for testing (format: 'YYYY-MM-DD')
    test_end_date : str
        End date for testing (format: 'YYYY-MM-DD')
    logistic_train_years : int
        Number of years to use for training logistic regression
    logistic_features : list
        Features for logistic regression
    stock_selection_method : str
        'top_n', 'threshold', or 'top_and_bottom'
    n_stocks : int
        Number of stocks to select
    lookback_window : int
        Number of months for nodewise regression rolling window
    transaction_cost : float
        Proportional transaction cost
    portfolio_type : str
        'gmv', 'mv', 'msr', or 'all' (compute all three portfolios)
    mv_target_variance : float
        Target variance for MV portfolio (default: 0.15^2/12)
    verbose : bool
        Print detailed logs
    
    Returns:
    --------
    results_dict : dict
        Dictionary with keys 'gmv', 'mv', 'msr' (depending on portfolio_type)
        Each contains: {'results_df': DataFrame, 'metrics': dict}
    """
    df = df.copy()
    df['datadate'] = pd.to_datetime(df['datadate'])
    
    all_dates = sorted(df['datadate'].unique())
    
    # Parse test dates
    test_start_dt = pd.to_datetime(test_start_date)
    test_end_dt = pd.to_datetime(test_end_date)
    
    # Extract years that need logistic regression retraining (every January in test period)
    start_year = test_start_dt.year
    end_year = test_end_dt.year
    test_years = list(range(start_year, end_year + 1))
    
    # Determine which portfolios to compute
    if portfolio_type == 'all':
        portfolio_types = ['gmv', 'mv', 'msr']
    else:
        portfolio_types = [portfolio_type]
    
    # Storage for each portfolio type
    results_storage = {ptype: {
        'returns': [],
        'dates': [],
        'weights_list': [],
        'turnover_list': [],
        'gross_returns': [],
        'prev_weights_dict': {},
        'prev_oos_returns_dict': {},
        'prev_gross_return': 0.0
    } for ptype in portfolio_types}
    
    # Track current stock universe
    current_permnos = []
    
    if verbose:
        print("="*70)
        print("INTEGRATED BACKTEST: LOGISTIC REGRESSION + NODEWISE")
        print(f"Test Period: {test_start_date} to {test_end_date}")
        print("="*70)
    
    for year in test_years:
        # Retrain logistic regression in January
        retrain_date = pd.to_datetime(f'{year}-01-31')
        
        if retrain_date not in all_dates:
            print(f"\n⚠ Warning: {retrain_date} not in dataset, skipping year {year}")
            continue
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"YEAR {year}: RETRAINING LOGISTIC REGRESSION")
            print(f"{'='*70}")
        
        # Define training window for logistic regression
        train_end = pd.to_datetime(f'{year-1}-12-31')
        train_start = pd.to_datetime(f'{year - rf_train_years}-01-31')
        
        if verbose:
            print(f"Logistic training period: {train_start.strftime('%Y-%m-%d')} to "
                  f"{train_end.strftime('%Y-%m-%d')}")
        
        # 1. Train Random Forest
        rf_model, scaler = train_random_forest(
            df, train_start, train_end, features=features
        )
        
        # 2. Select stocks using probability quantiles
        current_permnos = select_stocks_rf_quantiles(
            df, retrain_date, rf_model, scaler=scaler,
            features=features, top_q=top_q, bottom_q=bottom_q
        )
        
        if len(current_permnos) == 0:
            print(f"  ⚠ No stocks selected for {year}, skipping")
            continue
        
        # Determine date range for this year
        year_start_date = retrain_date
        
        # Determine end date for this year's strategy
        if year == end_year:
            # Last year: use test_end_date
            year_end_date = test_end_dt
        else:
            # Use December of current year
            year_end_date = pd.to_datetime(f'{year}-12-31')
            if year_end_date not in all_dates:
                # Find last available date in this year
                year_dates = [d for d in all_dates if d.year == year]
                year_end_date = max(year_dates) if year_dates else year_start_date
        
        # For the first year, respect test_start_date
        if year == start_year and test_start_dt > year_start_date:
            year_start_date = test_start_dt
        
        try:
            year_start_idx = all_dates.index(year_start_date)
            year_end_idx = all_dates.index(year_end_date)
        except ValueError as e:
            print(f"  ⚠ Date error: {e}")
            continue
        
        if verbose:
            print(f"\nRunning monthly rebalancing from {year_start_date.strftime('%Y-%m-%d')} "
                  f"to {year_end_date.strftime('%Y-%m-%d')}")
            print(f"{'='*70}")
        
        # Monthly loop for this year
        for t in range(year_start_idx, year_end_idx + 1):
            current_date = all_dates[t]
            
            if t < lookback_window:
                if verbose:
                    print(f"\n[{current_date.strftime('%Y-%m-%d')}] "
                          f"Skipping: insufficient lookback")
                continue
            
            # Define training window
            window_start_date = all_dates[t - lookback_window]
            window_end_date = all_dates[t - 1]
            
            # Filter data to selected stocks only
            train_data = df[
                (df['datadate'] >= window_start_date) & 
                (df['datadate'] <= window_end_date) &
                (df['permno'].isin(current_permnos))
            ]
            
            # Pivot returns
            returns_pivot = train_data.pivot(
                index='datadate', columns='permno', values='ret_fwd_1'
            )
            
            # Reindex to ensure all dates
            window_dates = all_dates[t - lookback_window : t]
            returns_pivot = returns_pivot.reindex(index=window_dates)
            
            # Filter assets with any NaNs
            nan_assets = returns_pivot.columns[returns_pivot.isna().any()]
            filtered_pivot = returns_pivot.drop(columns=nan_assets)
            
            current_assets = filtered_pivot.columns.tolist()
            Y = filtered_pivot.values
            n_train, p_current = Y.shape
            
            if verbose:
                month_num = t - year_start_idx + 1
                print(f"\n[Month {month_num}] {current_date.strftime('%Y-%m-%d')}")
                print(f"  Assets: {p_current}/{len(current_permnos)} with complete data")
            
            # Check validity
            if n_train < lookback_window or p_current < 2:
                if verbose:
                    print(f"  ⚠ Insufficient data, using previous weights")
                
                # Use previous weights for all portfolio types
                new_weights_dict = {ptype: results_storage[ptype]['prev_weights_dict'].copy() 
                                   for ptype in portfolio_types}
            else:
                try:
                    # Demean
                    Y_bar = Y.mean(axis=0)
                    Y_star = Y - Y_bar
                    
                    Theta_hat = sample_precision_matrix(Y)
                    
                    # Compute weights for each portfolio type
                    new_weights_dict = {}
                    
                    if 'gmv' in portfolio_types:
                        w_gmv = gmv_weights(Theta_hat)
                        new_weights_dict['gmv'] = {
                            asset: w_gmv[i] for i, asset in enumerate(current_assets)
                        }
                    
                    if 'mv' in portfolio_types or 'msr' in portfolio_types:
                        # Need expected returns (use sample mean from training window)
                        mu = Y_bar
                        
                        if 'mv' in portfolio_types:
                            w_mv = mv_weights(Theta_hat, mu, target_variance=mv_target_variance)
                            new_weights_dict['mv'] = {
                                asset: w_mv[i] for i, asset in enumerate(current_assets)
                            }
                        
                        if 'msr' in portfolio_types:
                            w_msr = msr_weights(Theta_hat, mu)
                            new_weights_dict['msr'] = {
                                asset: w_msr[i] for i, asset in enumerate(current_assets)
                            }
                    
                    if verbose:
                        print(f"  ✓ Nodewise completed for {', '.join(portfolio_types)}")
                    
                except Exception as e:
                    if verbose:
                        print(f"  ✗ Error: {e}")
                    new_weights_dict = {ptype: results_storage[ptype]['prev_weights_dict'].copy() 
                                       for ptype in portfolio_types}
            
            # Process each portfolio type
            for ptype in portfolio_types:
                # Normalize weights
                weights = new_weights_dict[ptype]
                weight_sum = sum(weights.values())
                if weight_sum > 1e-10:
                    weights = {k: v/weight_sum for k, v in weights.items()}
                else:
                    weights = results_storage[ptype]['prev_weights_dict'].copy()
            
            # Get OOS returns (common for all portfolio types)
            oos_data = df[df['datadate'] == current_date]
            oos_returns_series = oos_data.set_index('permno')['ret_fwd_1'].dropna()
            oos_returns_dict = oos_returns_series.to_dict()
            
            # Process each portfolio type
            for ptype in portfolio_types:
                # Get weights for this portfolio
                weights = new_weights_dict[ptype]
                prev_weights = results_storage[ptype]['prev_weights_dict']
                prev_oos_returns = results_storage[ptype]['prev_oos_returns_dict']
                prev_gross_ret = results_storage[ptype]['prev_gross_return']
                
                # Common assets
                common_assets = set(weights.keys()) & set(oos_returns_dict.keys())
                
                if len(common_assets) == 0:
                    continue
                
                # Filter and renormalize
                common_weights = {a: weights[a] for a in common_assets}
                common_weight_sum = sum(common_weights.values())
                if common_weight_sum > 1e-10:
                    common_weights = {k: v/common_weight_sum for k, v in common_weights.items()}
                else:
                    continue
                
                # Gross return
                gross_return = sum(
                    common_weights[a] * oos_returns_dict[a] for a in common_assets
                )
                
                if np.isnan(gross_return) or np.isinf(gross_return):
                    continue
                
                # Transaction costs
                if len(prev_weights) > 0:
                    adjusted_prev = {}
                    for asset, prev_w in prev_weights.items():
                        if asset in prev_oos_returns:
                            prev_r = prev_oos_returns[asset]
                            if abs(1 + prev_gross_ret) > 1e-6:
                                adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_ret)
                            else:
                                adjusted_prev[asset] = 0.0
                        else:
                            if abs(1 + prev_gross_ret) > 1e-6:
                                adjusted_prev[asset] = prev_w / (1 + prev_gross_ret)
                            else:
                                adjusted_prev[asset] = 0.0
                    
                    all_assets = set(adjusted_prev.keys()) | set(common_weights.keys())
                    turnover = sum(
                        abs(common_weights.get(a, 0.0) - adjusted_prev.get(a, 0.0))
                        for a in all_assets
                    )
                    tc = transaction_cost * (1 + gross_return) * turnover
                else:
                    turnover = sum(abs(w) for w in common_weights.values())
                    tc = transaction_cost * (1 + gross_return) * turnover
                
                net_return = gross_return - tc
                
                # Store results for this portfolio
                results_storage[ptype]['returns'].append(net_return)
                results_storage[ptype]['dates'].append(current_date)
                results_storage[ptype]['weights_list'].append(common_weights.copy())
                results_storage[ptype]['turnover_list'].append(turnover)
                results_storage[ptype]['gross_returns'].append(gross_return)
                
                # Update state
                results_storage[ptype]['prev_weights_dict'] = common_weights.copy()
                results_storage[ptype]['prev_oos_returns_dict'] = {a: oos_returns_dict[a] for a in common_assets}
                results_storage[ptype]['prev_gross_return'] = gross_return
            
            if verbose:
                # Print summary for all portfolios
                print(f"  Portfolio Returns:")
                for ptype in portfolio_types:
                    if len(results_storage[ptype]['returns']) > 0:
                        last_idx = len(results_storage[ptype]['returns']) - 1
                        gross_ret = results_storage[ptype]['gross_returns'][last_idx]
                        net_ret = results_storage[ptype]['returns'][last_idx]
                        to = results_storage[ptype]['turnover_list'][last_idx]
                        tc = gross_ret - net_ret
                        print(f"    {ptype.upper()}: Gross={gross_ret:>7.4f} | TO={to:>5.3f} | "
                              f"TC={tc:>7.5f} | Net={net_ret:>7.4f}")
    
    if verbose:
        print("\n" + "="*70)
        print("BACKTEST COMPLETE")
        print("="*70)
    
    # Compile results for each portfolio type
    results_dict = {}
    
    for ptype in portfolio_types:
        portfolio_returns = results_storage[ptype]['returns']
        portfolio_dates = results_storage[ptype]['dates']
        portfolio_turnover_list = results_storage[ptype]['turnover_list']
        portfolio_gross_returns = results_storage[ptype]['gross_returns']
        
        if len(portfolio_returns) == 0:
            results_dict[ptype] = {
                'results_df': pd.DataFrame(),
                'metrics': {}
            }
            continue
        
        results_df = pd.DataFrame({
            'date': portfolio_dates,
            'portfolio_return': portfolio_returns,
            'portfolio_gross_return': portfolio_gross_returns,
            'portfolio_turnover': portfolio_turnover_list
        })
        results_df['cumulative_return'] = (1 + results_df['portfolio_return']).cumprod() - 1
        
        # Overall metrics
        mean_return = np.mean(portfolio_returns)
        variance = np.var(portfolio_returns, ddof=1)
        sharpe_ratio = mean_return / np.sqrt(variance) if variance > 0 else 0
        
        annual_return = mean_return * 12
        annual_volatility = np.sqrt(variance * 12)
        annual_sharpe = annual_return / annual_volatility if annual_volatility > 0 else 0
        
        overall_metrics = {
            'mean_return': mean_return,
            'variance': variance,
            'sharpe_ratio': sharpe_ratio,
            'annual_return': annual_return,
            'annual_volatility': annual_volatility,
            'annual_sharpe_ratio': annual_sharpe,
            'total_return': results_df['cumulative_return'].iloc[-1],
            'avg_turnover': np.mean(portfolio_turnover_list),
            'n_periods': len(portfolio_returns)
        }
        
        results_dict[ptype] = {
            'results_df': results_df,
            'metrics': overall_metrics
        }
    
    return results_dict

In [4]:
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = (df.groupby('permno')['ret_excess'].shift(-1) )

In [5]:
variables = ['agr', 'bm', 'mom12m', 'mve', 'operprof', 'roeq', 'absacc',
            'acc', 'aeavol', 'age', 'baspread', 'BETA', 'bm_ia', 'cash',
            'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho',
            'chempia', 'chfeps', 'chinv', 'chmom', 'chnanalyst', 'chpmia',
            'chtx', 'cinvest', 'convind', 'currat', 'depr', 'disp', 'divi',
            'divo', 'dy', 'ear', 'egr', 'ep', 'fgr5yr', 'gma', 'grcapx',
            'grltnoa', 'herf', 'hire', 'idiovol', 'ill', 'indmom', 'invest',
            'IPO', 'lev', 'mom1m', 'mom36m', 'ms', 'mve_ia', 'nanalyst',
            'nincr', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr',
            'pchgm_pchsale', 'pchsale_pchinvt', 'pchsale_pchrect',
            'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'pricedelay', 'ps',
            'rd', 'rd_mve', 'rd_sale', 'realestate', 'retvol', 'roaq',
            'roavol', 'roic', 'rsup', 'salecash', 'saleinv', 'salerec',
            'secured', 'securedind', 'sfe', 'sgr', 'sin', 'sp', 'std_dolvol',
            'std_turn', 'stdcf', 'sue', 'tang', 'tb', 'turn', 'zerotrade']

In [6]:
results = integrated_backtest(
    df,
    test_start_date='2021-10-31',
    test_end_date='2024-04-30',
    rf_train_years=15,
    features=['mom12m', 'mve', 'bm'],
    top_q=0.90,
    bottom_q=0.10,
    lookback_window=180,
    transaction_cost=0.001,
    verbose=True
)

# Access results for each portfolio
gmv_results = results['gmv']['results_df']
gmv_metrics = results['gmv']['metrics']

mv_results = results['mv']['results_df']
mv_metrics = results['mv']['metrics']

msr_results = results['msr']['results_df']
msr_metrics = results['msr']['metrics']

# Compare Sharpe ratios
print(f"GMV Sharpe: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MV Sharpe:  {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MSR Sharpe: {msr_metrics['annual_sharpe_ratio']:.4f}")

INTEGRATED BACKTEST: LOGISTIC REGRESSION + NODEWISE
Test Period: 2021-10-31 to 2024-04-30

YEAR 2021: RETRAINING LOGISTIC REGRESSION
Logistic training period: 2006-01-31 to 2020-12-31
  Training samples: 89403
  Training R-squared: 0.0098
  Stocks evaluated: 497
  Upper expected return bound (Top 10%): 0.0090
  Lower expected return bound (Bottom 10%): 0.0064
  Stocks selected: 100

Running monthly rebalancing from 2021-10-31 to 2021-12-31

[Month 1] 2021-10-31
  Assets: 59/100 with complete data
  ✓ Nodewise completed for gmv, mv, msr
  Portfolio Returns:
    GMV: Gross=-0.0384 | TO=3.203 | TC=0.00308 | Net=-0.0415
    MV: Gross= 0.0021 | TO=7.977 | TC=0.00799 | Net=-0.0059
    MSR: Gross=-0.0096 | TO=6.021 | TC=0.00596 | Net=-0.0156

[Month 2] 2021-11-30
  Assets: 61/100 with complete data
  ✓ Nodewise completed for gmv, mv, msr
  Portfolio Returns:
    GMV: Gross= 0.0674 | TO=0.836 | TC=0.00089 | Net= 0.0665
    MV: Gross= 0.1143 | TO=1.943 | TC=0.00216 | Net= 0.1121
    MSR: Gross=

In [7]:
print(f"\n GMV")
print(f"Annualized Sharpe Ratio: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {gmv_metrics['mean_return']*12:.4f}")
print(f"Variance: {gmv_metrics['variance']*12:.4f}")
print(f"Avg Turnover: {gmv_metrics['avg_turnover']:.4f}")

print(f"\n MV")
print(f"Annualized Sharpe Ratio: {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {mv_metrics['mean_return']*12:.4f}")
print(f"Variance: {mv_metrics['variance']*12:.4f}")
print(f"Avg Turnover: {mv_metrics['avg_turnover']:.4f}")

print(f"\n MSR")
print(f"Annualized Sharpe Ratio: {msr_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {msr_metrics['mean_return']*12:.4f}")
print(f"Variance: {msr_metrics['variance']*12:.4f}")
print(f"Avg Turnover: {msr_metrics['avg_turnover']:.4f}")


 GMV
Annualized Sharpe Ratio: 0.0631
Mean Return: 0.0111
Variance: 0.0310
Avg Turnover: 1.1357

 MV
Annualized Sharpe Ratio: -0.2222
Mean Return: -0.0678
Variance: 0.0931
Avg Turnover: 2.8934

 MSR
Annualized Sharpe Ratio: -0.2116
Mean Return: -0.0548
Variance: 0.0670
Avg Turnover: 2.4259
